## Categorical Cross-Entropy loss derivative

The derivative is given as $ - \frac{y_{i,j}}{\hat{y}_{i,j}}$.

where i is the ith sample and j is the output label index.


In [ ]:
class Dense_layer:

    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    # we want to remember our input to calculate gradient of weights
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims = True)
        self.dinputs = np.dot(dvalues, self.weights.T)


class Activation_Relu:

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

import numpy as np
class Loss():
    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss
    
class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood


In [ ]:
class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues) # Number of samples

        #Number of labels in every sample
        #We'll use the first sample to count them
        labels = len(dvalues[0])

        #If labels are sparse then, turn then into one-hot vector
        if len(y_true.shape) == 1:
            y_true =   np.eye(labels)[y_true]

        #calculate gradient
        self.dinputs = -y_true / dvalues

        #Normalize gradient
        self.dinputs = self.dinputs / samples

## Softmax activation derivation

$S_{i,j}\delta_{j,k}\ -\ S_{i,j}S_{i,k}$

where $\delta$ is the kronecker delta.

In [ ]:
#A single sample
softmax_output = [0.7, 0.1, 0.2]

softmax_output = np.array(softmax_output).reshape(-1,1)
print(softmax_output)


In [ ]:
# Kronecker delta
print(np.eye(softmax_output.shape[0]))

In [ ]:
print(softmax_output * np.eye(softmax_output.shape[0]))

In [ ]:
# so the left side of the subtraction is 
print(np.diagflat(softmax_output))

In [ ]:
# the right side is given as 
print(np.dot(softmax_output, softmax_output.T))

In [ ]:
print(np.diagflat(softmax_output) - np.dot(softmax_output, softmax_output.T))

In [ ]:
class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1,1)

            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            #calculate sample-wise gradient
            #and add it to the array of sample gradients
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

## Common Categorical Cross-Entropy loss and softmax Activation Derivative

The drivative is given as $\hat{y_{i,k}}\ -\ y_{i,k}$

In [ ]:
class Activation_Softmax_Loss_CategoricalCrossentropy():

    def __init__(self):
        self.activation = Activation_softmax()
        self.loss = Loss_CategoricalCrossentropy()

    # Forward pass
    def forward(self, inputs, y_true):
        #output layer activation function
        self.activation.forward(inputs)

        #set the output
        self.output = self.activation.output

        #calculate and return loss value
        return self.loss.calculate(self.output, y_true)
    
    def backward(self, dvalues, y_true):

        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis = 1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs/samples



We can now test the class. We will use output of a softmax function for a batch of 3 sample.

In [ ]:
softmax_outputs = np.array([[0.7, 0.1, 0.2],
                            [0.1, 0.5, 0.4],
                            [0.02, 0.9, 0.08]])

class_targets = np.array([0, 1, 1])

In [ ]:
softmax_loss = Activation_Softmax_Loss_CategoricalCrossentropy()
softmax_loss.backward(softmax_outputs, class_targets)

In [ ]:
dvalues1 = softmax_loss.dinputs

In [ ]:
activation = Activation_softmax()
activation.output = softmax_outputs

In [ ]:
loss = Loss_CategoricalCrossentropy()

In [ ]:
loss.backward(softmax_outputs, class_targets)

In [ ]:
activation.backward(loss.dinputs)
dvalues2 = activation.dinputs

In [ ]:
print(dvalues1)
print(dvalues2)

Full Code now

In [ ]:
import numpy as np
from nnfs.datasets import vertical_data, spiral_data
import nnfs

In [ ]:
X, y = vertical_data(samples = 100, classes = 3)

# Dense layer with 2 input and 3 output values (3 neurons).
dense1 = Dense_layer(2,3)

activation1 = Activation_Relu()

#Create second dense layer that take output from previous layer hence 3 inputs and 3 output values
dense2 = Dense_layer(3,3)

#We are now using new class to calculate final softmax activation and loss calculation.
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)

loss = loss_activation.forward(dense2.output, y)

print(loss_activation.output[:5])

In [ ]:
print(loss)

In [ ]:
predictions = np.argmax(loss_activation.output, axis = 1)
if len(y.shape) == 2:
    y = np.argmax(y, axis = 1)
accuracy = np.mean(predictions == y)
print(accuracy)

In [ ]:
#Backward pass
loss_activation.backward(loss_activation.output, y)
dense2.backward(loss_activation.dinputs)
activation1.backward(dense2.dinputs)
dense1.backward(activation1.dinputs)

print(dense1.dweights)
print(dense1.dbiases)
print(dense2.dweights)
print(dense2.dbiases)